In [1]:
print("hello world")

hello world


In [3]:
import pandas as pd
import numpy as np
import time
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


import vitaldb as v

In [4]:
track_names = ["SNUADC/ART",        # Arterial pressure wave  | W/500 | mmHg
               "SNUADC/ECG_II",     # ECG lead II wave        | W/500 | mV
               "SNUADC/ECG_V5",     # ECG lead V5 wave        | W/500 | mV
               "BIS/EEG1_WAV",      # EEG wave from channel 1 | W/128 | uV
               "BIS/EEG2_WAV",      # EEG wave from channel 2 | W/128 | uV
               "Solar8000/RR_CO2",  # Respiratory rate based on capnography | N | /min
               "Primus/CO2",        # Capnography wave        | W/62.5 | mmHg
               "BIS/BIS",           # Bispectral index value  |    N   | unitless
               ]                

# read 1 patient from the VitalDB server
patient = v.VitalFile(1, track_names)  # should we add max length?
# convert to pandas dataframe
patient = patient.to_pandas(track_names=track_names, interval=5)
# change column names to 'human-readable'
patient.columns = ['arterial_pres', 'ecg1', 'ecg2', 'eeg1', 'eeg2', 'resp_rate', 'capnography', 'bis']

In [7]:
class TimeSeriesLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(TimeSeriesLSTM, self).__init__()

        # LSTM Layer
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        # Fully-connected Output Layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        out, _ = self.lstm(x) # _ : (hn, cn) are the final states

        out = out[:, -1, :]

        prediction = self.fc(out)
        return prediction
        

In [9]:
epochs = 20
batch_size = 32
seq_len = 200
split = 0.8
# device = 'cuda'
device = 'cpu'

patient = patient[['eeg1', 'eeg2', 'bis']].dropna()
patient = patient[patient['bis'] > 0]
patient['eeg1'] = (patient['eeg1'] - patient['eeg1'].mean()) / patient['eeg1'].std()
patient['eeg2'] = (patient['eeg2'] - patient['eeg2'].mean()) / patient['eeg2'].std()
x = np.stack([patient.eeg1, patient.eeg2], axis=1)
X = np.array([x[i:i+seq_len] for i in range(x.shape[0]-seq_len)])
X = X.reshape(-1, seq_len, 2)

y = patient.bis
y = (y - np.mean(y)) / np.std(y)
Y = np.array([y[i+seq_len] for i in range(len(y)-seq_len)]).reshape(-1, 1)

split_idx = int(len(X) * split)
X_train = X[:split_idx]
Y_train = Y[:split_idx]

model = TimeSeriesLSTM(2, 32, 3)
model.to(device)
model.train()

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

dataset_train = TensorDataset(torch.tensor(X_train, dtype=torch.float32).to(device), 
                        torch.tensor(Y_train, dtype=torch.float32).to(device))
dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)

X_val = X[split_idx:]
Y_val = Y[split_idx:]
dataset_val = TensorDataset(torch.tensor(X_val, dtype=torch.float32).to(device), 
                            torch.tensor(Y_val, dtype=torch.float32).to(device))
dataloader_val = DataLoader(dataset_val, batch_size=batch_size)

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch_x, batch_y in dataloader_train:
        optimizer.zero_grad()
        y_hat = model(batch_x)
        loss = criterion(y_hat, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation Check
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for v_x, v_y in dataloader_val:
            v_hat = model(v_x)
            v_loss = criterion(v_hat, v_y)
            val_loss += v_loss.item()
    
    print(f"Epoch {epoch+1}: Train Loss {train_loss/len(dataloader_train):.4f} | Val Loss {val_loss/len(dataloader_val):.4f}")

Epoch 1: Train Loss 0.7025 | Val Loss 1.2324
Epoch 2: Train Loss 0.6853 | Val Loss 1.2250
Epoch 3: Train Loss 0.6608 | Val Loss 1.3095
Epoch 4: Train Loss 0.5515 | Val Loss 1.8141
Epoch 5: Train Loss 0.4820 | Val Loss 2.2224
Epoch 6: Train Loss 0.4743 | Val Loss 1.6707
Epoch 7: Train Loss 0.4658 | Val Loss 2.4477
Epoch 8: Train Loss 0.4992 | Val Loss 2.3799
Epoch 9: Train Loss 0.4707 | Val Loss 2.3834
Epoch 10: Train Loss 0.4714 | Val Loss 2.3491
Epoch 11: Train Loss 0.4761 | Val Loss 1.5318
Epoch 12: Train Loss 0.4985 | Val Loss 2.4078
Epoch 13: Train Loss 0.4660 | Val Loss 2.3034
Epoch 14: Train Loss 0.4585 | Val Loss 1.6707
Epoch 15: Train Loss 0.4609 | Val Loss 1.6545
Epoch 16: Train Loss 0.4550 | Val Loss 2.4310
Epoch 17: Train Loss 0.4676 | Val Loss 1.7188
Epoch 18: Train Loss 0.4586 | Val Loss 2.5137
Epoch 19: Train Loss 0.4749 | Val Loss 1.6840
Epoch 20: Train Loss 0.4600 | Val Loss 2.3602


# ?????

In [2]:
import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

class EEGWindowDataset(Dataset):
    def __init__(self, input_dir, case_ids, seq_len, scaler=None, is_training=True):
        self.seq_len = seq_len
        self.patient_X = []
        self.patient_Y = []
        self.index_map = []  # Stores tuples of (patient_index, timestep_start)
        
        all_X_for_scaler = []
        
        print(f"Loading raw data into memory (Training={is_training})...")
        patient_idx = 0
        
        for cid in case_ids:
            sample_path = os.path.join(input_dir, f'case_{cid}.pt')
            if not os.path.exists(sample_path):
                continue
                
            data = torch.load(sample_path, weights_only=False)
            
            # Neural networks output NaN instantly if inputs have NaNs.
            # We replace any missing EEG features with 0.0 safely.
            x = np.nan_to_num(data['features'].numpy(), nan=0.0) 
            y = data['bis'].numpy()
            
            if x.shape[0] <= seq_len:
                continue
            
            self.patient_X.append(x)
            self.patient_Y.append(y)
            
            if is_training:
                all_X_for_scaler.append(x)
            
            # Pre-calculate valid indices! 
            # We only record the start index if the target Y at the END of the window is NOT NaN
            for start_t in range(x.shape[0] - seq_len):
                target_y = y[start_t + seq_len]
                if not np.isnan(target_y):
                    self.index_map.append((patient_idx, start_t))
                    
            patient_idx += 1

        # Fit and Apply Scaler on flat 2D data (super fast and memory efficient)
        if is_training and scaler is not None:
            print("Fitting standard scaler...")
            stacked_X = np.vstack(all_X_for_scaler)
            scaler.fit(stacked_X)
            
        if scaler is not None:
            print("Applying scaler...")
            for i in range(len(self.patient_X)):
                self.patient_X[i] = scaler.transform(self.patient_X[i])
                
        # Convert to tensors once to save CPU time during training
        self.patient_X = [torch.tensor(arr, dtype=torch.float32) for arr in self.patient_X]
        self.patient_Y = [torch.tensor(arr, dtype=torch.float32) for arr in self.patient_Y]
        print(f"Dataset ready. Total valid 60s windows: {len(self.index_map)}")

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        # 1. Look up which patient and which timestamp this index corresponds to
        p_idx, start_t = self.index_map[idx]
        
        # 2. Slice the 60-step window from the raw data ON THE FLY
        X_window = self.patient_X[p_idx][start_t : start_t + self.seq_len]
        
        # 3. Grab the target BIS value at the end of the window
        Y_target = self.patient_Y[p_idx][start_t + self.seq_len]
        
        # Return X and Y (unsqueeze Y to make it shape [1] instead of a scalar)
        return X_window, Y_target.unsqueeze(0)

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
import wandb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# =====================================================================
# 1. DATA LOADING FUNCTION (Unchanged)
# =====================================================================
def load_pt_samples(input_dir, case_ids, seq_len):
    """Loads 3D sliding windows directly from .pt files for specific patients."""
    X_list = []
    Y_list = []

    for cid in case_ids:
        sample_path = os.path.join(input_dir, f'case_{cid}.pt')
        if not os.path.exists(sample_path):
            continue
            
        data = torch.load(sample_path, weights_only=False)
        x = data['features'].numpy()
        y = data['bis'].numpy()
            
        num_samples = x.shape[0]
        if num_samples <= seq_len:
            continue
            
        # Create sliding windows per patient
        X_case = np.array([x[i:i+seq_len] for i in range(num_samples - seq_len)])
        Y_case = np.array([y[i+seq_len] for i in range(num_samples - seq_len)]).reshape(-1, 1)
        
        X_list.append(X_case)
        Y_list.append(Y_case)

    # Combine all patients
    X = np.concatenate(X_list, axis=0)
    Y = np.concatenate(Y_list, axis=0)
    
    return X, Y

# =====================================================================
# 2. LSTM MODEL DEFINITION
# =====================================================================
class BISPredictorLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size=1, dropout=0.2):
        super(BISPredictorLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # batch_first=True means input shape should be (batch, seq_len, features)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x shape: (batch_size, seq_len, features)
        out, _ = self.lstm(x)
        
        # We only want the output from the final time step in the sequence
        # out shape becomes: (batch_size, hidden_size)
        out = out[:, -1, :] 
        
        # Pass through the linear layer to get the final BIS prediction
        out = self.fc(out)
        return out

# =====================================================================
# 3. MAIN EXECUTION PIPELINE
# =====================================================================
# Configuration
INPUT_DIR = '../../data/processed/eeg'
CASES_FILE = '../../data/processed/cases_data.csv'
SEQ_LEN = 60 # 60-second window

# Device configuration (use GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

# A. Load Patient IDs and Split
cases_master = pd.read_csv(CASES_FILE)
all_ids = cases_master['caseid'].tolist()
train_ids, test_ids = train_test_split(all_ids, test_size=0.2, random_state=42)

# B. Initialize Scaler and PyTorch Datasets
scaler = StandardScaler()

# The training dataset will FIT the scaler
train_dataset = EEGWindowDataset(
    input_dir=INPUT_DIR, 
    case_ids=train_ids, 
    seq_len=SEQ_LEN, 
    scaler=scaler, 
    is_training=True
)

# The testing dataset will USE the fitted scaler
test_dataset = EEGWindowDataset(
    input_dir=INPUT_DIR, 
    case_ids=test_ids, 
    seq_len=SEQ_LEN, 
    scaler=scaler, 
    is_training=False
)

# C. Prepare DataLoaders
BATCH_SIZE = 256
# num_workers=4 will use multiple CPU cores to slice the windows in parallel
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE, num_workers=4)
test_loader  = DataLoader(test_dataset, shuffle=False, batch_size=BATCH_SIZE, num_workers=4)

# D. Define Model Parameters & Initialize W&B
params = {
    'input_size': train_dataset.patient_X[0].shape[-1], # Grabs the feature count dynamically
    'hidden_size': 64,                     
    'num_layers': 2,                       
    'learning_rate': 0.001,
    'epochs': 30,
    'batch_size': BATCH_SIZE
}

wandb.init(project="eeg-bis-prediction", config=params, name="lstm-baseline")

model = BISPredictorLSTM(
    input_size=params['input_size'], 
    hidden_size=params['hidden_size'], 
    num_layers=params['num_layers']
).to(device)

criterion = nn.MSELoss() # Same as 'reg:squarederror' in XGBoost
optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'])

# E. Train the Model
print("\nTraining LSTM Model...")
for epoch in range(params['epochs']):
    model.train()
    train_loss = 0.0
    
    for batch_X, batch_Y in train_loader:
        batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)
        
        # Forward pass
        predictions = model(batch_X)
        loss = criterion(predictions, batch_Y)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_X.size(0)
        
    train_loss = train_loss / len(train_loader.dataset)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_Y in test_loader:
            batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)
            preds = model(batch_X)
            loss = criterion(preds, batch_Y)
            val_loss += loss.item() * batch_X.size(0)
            
    val_loss = val_loss / len(test_loader.dataset)
    val_rmse = np.sqrt(val_loss)
    
    print(f"Epoch [{epoch+1}/{params['epochs']}] | Train MSE: {train_loss:.4f} | Val RMSE: {val_rmse:.4f}")
    
    # Log to W&B
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_rmse": val_rmse
    })

# F. Final Evaluation
print("\n--- FINAL RESULTS ---")
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for batch_X, batch_Y in test_loader:
        batch_X = batch_X.to(device)
        preds = model(batch_X).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(batch_Y.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

mae = mean_absolute_error(all_targets, all_preds)
rmse = np.sqrt(mean_squared_error(all_targets, all_preds))

print(f"Mean Absolute Error (MAE): {mae:.2f} BIS points")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} BIS points")

wandb.log({
    "final_test_mae": mae,
    "final_test_rmse": rmse
})

wandb.finish()

Using device: cpu
Loading raw data into memory (Training=True)...
Fitting standard scaler...
Applying scaler...
Dataset ready. Total valid 60s windows: 1487159
Loading raw data into memory (Training=False)...


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\mikol\_netrc.


Applying scaler...
Dataset ready. Total valid 60s windows: 380202


wandb: Currently logged in as: mik-nowacki (mik-nowacki-chalmers-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Training LSTM Model...
